<img src="logo.png" alt="Vegeta" width="240">

# Kestrel benchmark — the family on one table

The results of the design notebooks side by side, read from what they exported: **43** (Kestrel-4Q-1/2/3, the biplanes),
**45** (Kestrel-X-A/B, the X-wings) and **46** (Kestrel-Q4T, the tandems). Nothing is computed here; a notebook that has
not run leaves its rows out and says so. Run the design notebooks first, then this one.

In [ ]:
import os, sys, time
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
sys.path.insert(0, "designs")
import kestrel_benchmark as kb
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 40)
from vegeta import yanus
GATE = yanus.enter("47_kestrel_benchmark")       # the door keeper: the three design files are optional takes (yanus_auto: produced when missing)
display(GATE.table())

## 1. The table

In [ ]:
DF, NOTES = kb.load()
display(DF.T)
print("\n".join("· " + n for n in NOTES))

## 2. Ranks and the two measures the mission decides on

Rank 1 is best per measure (lower mass, drag, stall speed and power; higher lift, endurance and authority); the mean rank
is only a reading aid. The chart: the cruise power at 16 m/s against the hover pitch authority — an aircraft that cruises
pays the left bar, one that hovers and transitions needs the right one.

In [ ]:
RANKS = kb.ranks(DF)
display(RANKS)
fig, ax = plt.subplots(1, 2, figsize=(14, 0.45 * len(DF) + 1.5), sharey=True)
kb.plot(DF, ax=ax); plt.tight_layout(); plt.show()

## 3. The methodology: points 1–10 per category, weights per mission

Each category is one measure of the table scored on a straight line between two **anchors**: 1 point at the brief's
worst acceptable value, 10 at its target, clipped outside (`kestrel_benchmark.CATEGORIES`). The anchors are design
targets, not the family's extremes, so a score means the same thing when an airframe is added. Where a notebook
exported no number, the category takes the family's constant from the notebooks' text (`FAMILY`: ground handling,
payload fit, evidence) and the table says so. A **mission** is a weight set over the categories (`MISSIONS`, each
summing to 1); the weighted total leaves a blank category out and renormalises, printing the coverage. Edit the anchors
and the weights in `designs/kestrel_benchmark.py`; nothing else in the notebook changes.

In [ ]:
display(kb.anchors_table())
display(pd.DataFrame(kb.MISSIONS).rename_axis("weights by mission"))
display(pd.DataFrame({f: {k: v for k, v in d.items() if k != "notes"} for f, d in kb.FAMILY.items()}).T.rename_axis("family constants (0–1), from the notebooks' text"))
print("\n".join(f"· {f}: {d['notes']}" for f, d in kb.FAMILY.items()))
SC = kb.scores(DF)
display(SC.round(1).rename_axis("points 1–10"))
fig, ax = plt.subplots(figsize=(15, 5)); kb.plot_scores(SC, ax=ax); plt.tight_layout(); plt.show()
WT = kb.weighted(SC)
display(WT[[c for c in WT.columns if "coverage" not in c]].round(2).rename_axis("weighted total 1–10 per mission"))
fig, ax = plt.subplots(1, len(kb.MISSIONS), figsize=(4.2 * len(kb.MISSIONS), 0.45 * len(DF) + 1.8)); kb.plot_missions(WT, ax=ax); plt.tight_layout(); plt.show()
for m in kb.MISSIONS:
    s = pd.to_numeric(WT[m], errors="coerce").sort_values(ascending=False)
    print(f"{m}: " + ", ".join(f"{i.replace('Kestrel-', '')} {v:.1f}" for i, v in s.head(3).items()))

## 4. Export

In [ ]:
import kestrel_report as kr
EXP = kr.export("47_kestrel_benchmark", "Kestrel benchmark — the family on one table and the mission scores (from notebooks 43, 45, 46)",
                {"benchmark": DF, "ranks": RANKS, "anchors": kb.anchors_table(), "mission weights": pd.DataFrame(kb.MISSIONS), "scores 1-10": SC, "weighted by mission": WT, "notes": {str(i): n for i, n in enumerate(NOTES)}})
STAMPS = GATE.leave()
print(f"{len(STAMPS)} output(s) stamped")